# End-to-End Pipeline Evaluation

Runs the full pipeline (raw text → NER → sentiment) on the benchmark dataset
(`data/labeled/final/holdout_relabeled.jsonl`) and saves metrics + summary to
`outputs/e2e_evaluation/` on Drive.

Unlike the Stage 3 holdout evaluation, this does NOT use gold entity positions —
the NER head predicts spans first, then the sentiment head scores those predictions,
and metrics are computed against gold via char-level IoU matching.

**Outputs (timestamped):**
- `e2e_metrics_<ts>.json` — full metrics (NER F1 per type, coverage, sentiment, joint accuracy, bucket confusion)
- `e2e_summary_<ts>.txt` — human-readable summary
- `e2e_eval_<ts>.log` — run log
- `e2e_predictions_<ts>.jsonl` — (optional) per-article predictions for error analysis

Runtime auto-terminates at the end to stop billing.

In [ ]:
# 1. Mount Drive & check GPU
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Set project path & choose checkpoint
import os
PROJECT_PATH = "/content/drive/MyDrive/entity_sentiment_model_pipeline"

# NEW dropout-trained Stage 3 model (May 2026 retrain).
# IMPORTANT: best_model.pt in the v2retrain dir is STALE (contains epoch 1
# weights, corr=0.3274) due to a Drive-sync issue during training. The actual
# best is checkpoint_epoch_9.pt (corr=0.5105). Use that directly.
CHECKPOINT_PATH = f"{PROJECT_PATH}/checkpoints/stage3_sentiment_large_v2retrain/checkpoint_epoch_9.pt"

# To run the OLD Apr-18 model (pre-dropout-fix) for comparison, use:
# CHECKPOINT_PATH = f"{PROJECT_PATH}/checkpoints/best_model_20260418.pt"

assert os.path.exists(PROJECT_PATH), f"Not found: {PROJECT_PATH}"
assert os.path.exists(f"{PROJECT_PATH}/scripts/evaluation/evaluate_e2e_pipeline.py"), "evaluate_e2e_pipeline.py not found!"
assert os.path.exists(f"{PROJECT_PATH}/data/labeled/final/holdout_relabeled.jsonl"), "benchmark file not found!"
assert os.path.exists(CHECKPOINT_PATH), f"Checkpoint not found: {CHECKPOINT_PATH}"

# Verify we have the right epoch loaded — guard against the stale-checkpoint bug
import torch
_ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
_corr = _ckpt.get("val_metrics", {}).get("sentiment_corr", 0)
_epoch = _ckpt.get("epoch", -1) + 1  # zero-indexed -> human
print(f"Project    : {PROJECT_PATH}")
print(f"Checkpoint : {CHECKPOINT_PATH}")
print(f"  epoch    : {_epoch}  (val Pearson r = {_corr:.4f})")
print(f"Benchmark  : .../data/labeled/final/holdout_relabeled.jsonl  (6,750 articles)")
assert _corr > 0.45, f"Checkpoint looks stale (corr={_corr:.4f}) — expected > 0.45"
del _ckpt

In [ ]:
# 3. Install deps
!pip install -q transformers torch torchvision torchaudio
!pip install -q pytorch-crf

In [ ]:
# 4. Run end-to-end evaluation
#    --ner-mode single-pass: the new dropout-trained model handles CLS-only encoder
#                            natively. No need for the iterative 2-pass NER hack.
#    --inference-batch-size 16: batches articles through the encoder for ~6-8x speedup.
#                            On 95GB G4 with no_grad, 16 is safe (peak ~3-5 GB).
#                            If you see CUDA OOM, drop to 8 or 4.
#    --save-predictions writes per-article details (~50-100 MB for 6,750 articles).

!cd {PROJECT_PATH} && python scripts/evaluation/evaluate_e2e_pipeline.py \
    --checkpoint {CHECKPOINT_PATH} \
    --benchmark {PROJECT_PATH}/data/labeled/final/holdout_relabeled.jsonl \
    --output-dir {PROJECT_PATH}/outputs/e2e_evaluation \
    --local-output-dir /content \
    --ner-mode single-pass \
    --inference-batch-size 16 \
    --iou-threshold 0.5 \
    --max-length 2048 \
    --log-every 100 \
    --save-predictions

In [ ]:
# 5. Display the results (latest run)
import os, json, glob

# Try Drive first; fall back to local /content if Drive is still syncing
out_drive = f"{PROJECT_PATH}/outputs/e2e_evaluation"
out_local = "/content"

summaries = sorted(glob.glob(f"{out_drive}/e2e_summary_*.txt"))
if not summaries:
    summaries = sorted(glob.glob(f"{out_local}/e2e_summary_*.txt"))

metrics_files = sorted(glob.glob(f"{out_drive}/e2e_metrics_*.json"))
if not metrics_files:
    metrics_files = sorted(glob.glob(f"{out_local}/e2e_metrics_*.json"))

if summaries:
    print(f"Latest summary: {summaries[-1]}\n")
    with open(summaries[-1]) as f:
        print(f.read())
else:
    print("No summary file found yet — evaluation may still be running or failed.")

print('\n' + '='*80)
print('Files written:')
for d in [out_drive, out_local]:
    if os.path.exists(d):
        for f in sorted(os.listdir(d)):
            if f.startswith('e2e_'):
                size = os.path.getsize(os.path.join(d, f)) / 1e6
                print(f"  {d}/{f}  ({size:.2f} MB)")

In [ ]:
# 6. (Optional) Quick visualization of headline metrics
import json, glob
metrics_files = sorted(glob.glob(f"{PROJECT_PATH}/outputs/e2e_evaluation/e2e_metrics_*.json"))
if not metrics_files:
    metrics_files = sorted(glob.glob('/content/e2e_metrics_*.json'))

if metrics_files:
    with open(metrics_files[-1]) as f:
        m = json.load(f)

    print('Headline:')
    print(f"  Articles            : {m['n_articles']:,}")
    print(f"  Elapsed             : {m['elapsed_seconds']:.0f}s ({m['elapsed_seconds']/60:.1f}min)")
    print(f"  Sentiment-type F1   : {m['ner_metrics']['overall_sentiment_types']['f1']:.4f}")
    print(f"  Coverage            : {m['coverage']['coverage_rate']:.4f}")
    sm = m['sentiment_on_covered']
    if sm:
        print(f"  Sentiment Pearson r : {sm['pearson_r']:.4f}")
        print(f"  Sentiment MAE       : {sm['mae']:.4f}")
        print(f"  Joint acc tol=0.2   : {m['joint_accuracy']['tol_0.2']:.4f}")
        print(f"  Joint acc tol=0.4   : {m['joint_accuracy']['tol_0.4']:.4f}")
    print()
    print('Per-type NER F1:')
    for t, mt in sorted(m['ner_metrics']['per_type'].items()):
        print(f"  {t:<10}  F1={mt['f1']:.4f}  (P={mt['precision']:.4f} R={mt['recall']:.4f}  TP={mt['tp']:,} FP={mt['fp']:,} FN={mt['fn']:,})")
    print()
    print('Per-type sentiment (on covered):')
    for t, mt in sorted(m['per_type_sentiment'].items()):
        print(f"  {t:<10}  N={mt['n']:,}  MSE={mt['mse']:.4f}  MAE={mt['mae']:.4f}  Corr={mt['pearson_r']:.4f}")

In [ ]:
# 7. Terminate runtime to stop billing (run when you're done inspecting results)
from google.colab import drive, runtime
try:
    drive.flush_and_unmount()
except Exception as e:
    print(f'flush_and_unmount: {e}')
runtime.unassign()